In [1]:
from zul.utilities.redis_vector_helper import RedisVectorDB
import numpy as np

In [ ]:
redis_db = RedisVectorDB(
    host="0.0.0.0",
    port=8000,
    password="password"
)

15:36:35 zul.utilities.redis_vector_helper INFO   Successfully connected to Redis at 10.14.204.202:6006


In [3]:
# 2. CREATE INDEX DENGAN SCHEMA
schema = {
    "index": {
        "name": "user_simple",
        "prefix": "user_simple_docs",
    },
    "fields": [
        {"name": "user", "type": "tag"},
        {"name": "credit_score", "type": "tag"},
        {"name": "job", "type": "text"},
        {"name": "age", "type": "numeric"},
        {
            "name": "user_embedding",
            "type": "vector",
            "attrs": {
                "dims": 3,
                "distance_metric": "cosine",
                "algorithm": "flat",
                "datatype": "float32"
            }
        }
    ]
}


In [4]:
redis_db.create_index(schema, overwrite=True)
# Check requirements
requirements = redis_db.check_requirements()
print(f"Requirements: {requirements}")

15:37:02 redisvl.index.index INFO   Index already exists, overwriting.
15:37:02 zul.utilities.redis_vector_helper INFO   Index 'user_simple' created successfully
Versi Redis: 7.4.5
[{b'name': b'search', b'ver': 21020, b'path': b'/opt/redis-stack/lib/redisearch.so', b'args': []}]
Modul Search (RediSearch) tidak ditemukan. Instal Redis Stack atau load modul secara manual.
Requirements: None


In [5]:
# 3. INSERT DATA
data = [
    {
        'user': 'john',
        'age': 1,
        'job': 'engineer',
        'credit_score': 'high',
        'user_embedding': np.array([0.1, 0.1, 0.5], dtype=np.float32).tobytes()
    },
    {
        'user': 'mary',
        'age': 2,
        'job': 'doctor',
        'credit_score': 'low',
        'user_embedding': np.array([0.1, 0.1, 0.5], dtype=np.float32).tobytes()
    },
    {
        'user': 'joe',
        'age': 3,
        'job': 'dentist',
        'credit_score': 'medium',
        'user_embedding': np.array([0.9, 0.9, 0.1], dtype=np.float32).tobytes()
    }
]

keys = redis_db.insert_data(data)
print(f"Inserted keys: {keys}")

15:37:20 zul.utilities.redis_vector_helper INFO   Inserted batch 1: 3 records
15:37:20 zul.utilities.redis_vector_helper INFO   Total 3 records inserted successfully
Inserted keys: ['user_simple_docs:01K99JKMDN63RP97CBKP5KF3RC', 'user_simple_docs:01K99JKMDTCW18NEKQBQGJRC7E', 'user_simple_docs:01K99JKMDVD3H1MHJG66FR7JAR']


In [6]:
# 4. SEARCH (VECTOR SEARCH)
query_vector = np.array([0.1, 0.1, 0.5], dtype=np.float32)

results = redis_db.vector_search(
    vector_query=query_vector,
    vector_field_name="user_embedding",
    return_fields=["user", "job", "age", "credit_score"],
    num_results=2
)
results

15:37:30 zul.utilities.redis_vector_helper INFO   Vector search completed: 2 results found


[{'id': 'user_simple_docs:01K99HPW9PY52DW63HV65QHGJB',
  'vector_distance': '0',
  'user': 'john',
  'job': 'engineer',
  'age': '1',
  'credit_score': 'high'},
 {'id': 'user_simple_docs:01K99HH8QEH6WMBQ8FXYN6K6AR',
  'vector_distance': '0',
  'user': 'mary',
  'job': 'doctor',
  'age': '2',
  'credit_score': 'low'}]

In [8]:
# 4. SEARCH (VECTOR SEARCH)
query_vector = np.array([0.1, 0.1, 0.5], dtype=np.float32)

results = redis_db.hybrid_search(
        text_query= "hallo",
        vector_query=query_vector,
        text_field_name="user",
        vector_field_name="user_embedding",
        text_scorer="BM25",
        return_fields=["user", "job", "age", "credit_score"],
        num_results= 2
)
results

15:39:38 zul.utilities.redis_vector_helper INFO   Hybrid search completed: 2 results found


[{'vector_distance': '0',
  'user': 'mary',
  'job': 'doctor',
  'age': '2',
  'credit_score': 'low',
  'vector_similarity': '1',
  'text_score': '0',
  'hybrid_score': '0.7'},
 {'vector_distance': '0',
  'user': 'john',
  'job': 'engineer',
  'age': '1',
  'credit_score': 'high',
  'vector_similarity': '1',
  'text_score': '0',
  'hybrid_score': '0.7'}]

In [9]:

# 4. SEARCH 
query_vector = np.array([0.1, 0.1, 0.5], dtype=np.float32)

results = redis_db.hybrid_search_rrf(
        text_query= "hallo",
        vector_query=query_vector,
        text_field_name="user",
        vector_field_name="user_embedding",
        text_scorer="BM25",
        return_fields=["user", "job", "age", "credit_score"],
        num_results= 2
)
results

([('john', 0.009900990099009901), ('mary', 0.00980392156862745)],
 ['john', 'mary'])